In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_matching(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
            # RealEstateMatchingId String, SourceSystem String, IngestionTimestamp timestamp, BatchId String, RealEstateMatchingKey String, RealEstateKey int, PorfolioKey String, PorfolioName String, MatchedRealEstateKey String, NewRealEstateKey String, RealEstateMatchingCompletedBy int, RealEstateMatchingCompletedDate timestamp, RealEstateMatchingApprovedBy int, RealEstateMatchingApprovedDate timestamp, RealEstateMatchingLawyerCode int, RealEstateMatchingLawyer String, RealEstateResearchPerson int, RealEstateCodeByLawyers String, RealEstateMatchingStatus String, RealEstateMatchingApprovedStatus String, RealEstateMatchingReviewedStatus String, RealEstateBuildingCodeByLawyers String, BusinessDate date
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateMatching"
        prpropertyTemp_extra_filters = None

        if prpropertyTemp_extra_filters is not None:
            prpropertyTemp_extra_filters.append(date_filter)
            prpropertyTemp_final_filters = " AND ".join(prpropertyTemp_extra_filters)
        else:
            prpropertyTemp_final_filters = date_filter


        prpropertyTemp_selected_fields = [
            "id",
            "portfolio_code",
            "matched_property",
            "new_property",
            "user_completed",
            "date_completed",
            "approver",
            "approved_date",
            "lawyer_id",
            "lawyer_code",
            "contact_id",
            "lawyer_property",
            "matching_status",
            "matching_approval",
            "review_status",
            "businessdate"
        ]

        prpropertytemp = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prpropertytemp",
            prpropertyTemp_final_filters,
            prpropertyTemp_selected_fields,
        )

        prpropertytemp = prpropertytemp.withColumnRenamed("id", "prpropertytemp_id")

        # ================== prpropertybuildingtemp ==================

        prpropertybuildingtemp_extra_filters = None

        if prpropertybuildingtemp_extra_filters is not None:
            prpropertybuildingtemp_extra_filters.append(date_filter)
            prpropertybuildingtemp_final_filters = " AND ".join(
                prpropertybuildingtemp_extra_filters
            )
        else:
            prpropertybuildingtemp_final_filters = date_filter

        prpropertybuildingtemp_selected_fields = [
            "id",
            "lawyer_building",
            "property_temp_id"
        ]

        prpropertybuildingtemp = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prpropertybuildingtemp",
            prpropertybuildingtemp_final_filters,
            prpropertybuildingtemp_selected_fields,
        )

        prpropertybuildingtemp = prpropertybuildingtemp.withColumnRenamed("id", "prpropertybuildingtemp_id")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix"
        ]

        print(curated_zone)
        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Joins ==================

        prpropertybuildingtemp_join_condition = (
            prpropertytemp["prpropertytemp_id"] == prpropertybuildingtemp["property_temp_id"]
        )

        prproperty_table = prpropertytemp.join(
            prpropertybuildingtemp, prpropertybuildingtemp_join_condition, "left"
        )

        portfolio_join_condition = (
            prproperty_table["portfolio_code"] == portfolio["Prefix"]
        )

        target_table = prproperty_table.join(
            portfolio, portfolio_join_condition, "inner"
        )

        target_table_primary_key_fields = [
            target_table.prpropertytemp_id,
            target_table.prpropertybuildingtemp_id,
            lit("thesis_RES"),
        ]
        RealEstateMachingKey_primary_key_fields = [
            target_table.prpropertytemp_id,
            target_table.prpropertybuildingtemp_id,
        ]

        target_table = target_table.withColumn(
            "RealEstateMatchingId",
            sha2(concat_ws(";", *target_table_primary_key_fields), 256),
        ).withColumn(
            "RealEstateInfoKey", concat_ws(";", *RealEstateMachingKey_primary_key_fields)
        )


        result_df = target_table.select(
            target_table.RealEstateMatchingId,
            lit("thesis_RES").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table.RealEstateMatchingId.alias("RealEstateMatchingKey"),
            target_table.prpropertytemp_id.alias("RealEstateKey"),
            target_table.portfolio_code.alias("PorfolioKey"),
            target_table.PortfolioName.alias("PorfolioName"),
            target_table.matched_property.alias("MatchedRealEstateKey"),
            target_table.new_property.alias("NewRealEstateKey"),
            target_table.user_completed.alias("RealEstateMatchingCompletedBy"),
            target_table.date_completed.alias("RealEstateMatchingCompletedDate"),
            target_table.approver.alias("RealEstateMatchingApprovedBy"),
            target_table.approved_date.alias("RealEstateMatchingApprovedDate"),
            target_table.lawyer_id.alias("RealEstateMatchingLawyerCode"),
            target_table.lawyer_code.alias("RealEstateMatchingLawyer"),
            target_table.contact_id.alias("RealEstateResearchPerson"),
            target_table.lawyer_property.alias("RealEstateCodeByLawyers"),
            target_table.matching_status.alias("RealEstateMatchingStatus"),
            target_table.matching_approval.alias("RealEstateMatchingApprovedStatus"),
            target_table.review_status.alias("RealEstateMatchingReviewedStatus"),
            target_table.lawyer_building.alias("RealEstateBuildingCodeByLawyers"),
            target_table.businessdate.alias("BusinessDate")
        )
        # ================== Append ==================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

   # logging.info("We did it!")


In [0]:
load_curated_real_estate_matching(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)